# evaluate demo

A clean walkthrough of the new two-step anonymization workflow:

1. `anonymizer.preview(...)` (or `.run(...)`) does **detection + replacement only** — no LLM judges, no evaluation cost.
2. `anonymizer.evaluate(preview)` is a **separate, opt-in** call that scores the result with four LLM-as-judge metrics:
   - **Detection Validity** — were the detected entities correct?
   - **Type Fidelity** — does each synthetic preserve the entity class/format? *(Substitute only)*
   - **Relational Consistency** — do cross-entity relationships stay coherent? *(Substitute only)*
   - **Attribute Fidelity** — do salient within-entity attributes (gender, age bucket) survive? *(Substitute only)*

For non-Substitute strategies (Annotate / Redact / Hash), only Detection Validity runs — the other three need a replacement map.

The split lets you iterate on judge prompts/models without re-running the (expensive) detection + replacement step every time.

The strategy is carried on the `preview` / `result` object itself, so `evaluate(preview)` knows what to dispatch — you don't restate the mode.

## Provider setup

Models / endpoints used in this notebook. The API key for `nvidia` is read from `INTERNAL_NVIDIA_API_KEY`; uncomment the `getpass` block below if it isn't already in your environment.

In [ ]:
import getpass
import os

from data_designer.config.models import ModelProvider

GLINER_BREV_ENDPOINT = "https://gliner-qaqtckhiy.brevlab.com/v1"
NVIDIA_INFERENCE_ENDPOINT = "https://inference-api.nvidia.com"

MODEL_CONFIGS = """
model_configs:
  - alias: gliner-pii-detector
    model: nvidia/gliner-pii
    provider: nvidia/gliner-pii
    inference_parameters:
      max_parallel_requests: 1
      timeout: 120

  - alias: gpt-oss-120b
    model: nvidia/openai/gpt-oss-120b
    provider: nvidia
    inference_parameters:
      max_parallel_requests: 16
      max_tokens: 16384
      temperature: 0.3
      top_p: 0.95
      timeout: 300

  - alias: nemotron-30b-thinking
    model: nvidia/nvidia/Nemotron-3-Nano-30B-A3B
    provider: nvidia
    inference_parameters:
      max_parallel_requests: 16
      max_tokens: 8192
      temperature: 0.4
      top_p: 1.0
      timeout: 300
"""


def get_providers() -> list[ModelProvider]:
    if not os.getenv("INTERNAL_NVIDIA_API_KEY"):
        key = getpass.getpass("Enter INTERNAL_NVIDIA_API_KEY: ").strip()
        if not key:
            raise RuntimeError("INTERNAL_NVIDIA_API_KEY is required for the LLM roles.")
        os.environ["INTERNAL_NVIDIA_API_KEY"] = key
    return [
        ModelProvider(
            name="nvidia/gliner-pii",
            endpoint=GLINER_BREV_ENDPOINT,
            provider_type="openai",
        ),
        ModelProvider(
            name="nvidia",
            endpoint=NVIDIA_INFERENCE_ENDPOINT,
            provider_type="openai",
            api_key="INTERNAL_NVIDIA_API_KEY",
        ),
    ]

In [ ]:
from anonymizer import Anonymizer, AnonymizerConfig, AnonymizerInput, Redact, Substitute

## Initialize the Anonymizer

In [ ]:
anonymizer = Anonymizer(
    model_configs=MODEL_CONFIGS,
    model_providers=get_providers(),
)

## Input data

Same dataset used as the running example.

In [ ]:
input_data = AnonymizerInput(
    source="/Users/memadi/Anonymizer/test-data/NVIDIA_synthetic_biographies.csv",
    text_column="biography",
    data_summary="Biographical profiles",
)
input_data

## Step 1 — Anonymize (detection + replacement only)

This step runs the GLiNER detector, validation / augmentation, and (for Substitute) the LLM that generates the replacement map. **No evaluation judges fire here**, so this is the cheap step you can iterate on freely.

In [ ]:
substitute_config = AnonymizerConfig(replace=Substitute())

substitute_preview = anonymizer.preview(
    config=substitute_config,
    data=input_data,
    num_records=15,
)

In [ ]:
# Render one record — replaced text, entity highlights, replacement table.
# No LLM judgee yet because we haven't evaluated.
substitute_preview.display_record(0)

## Step 2 — Evaluate replacement quality

`anonymizer.evaluate(output)` runs the four judges as columns of a **single DataDesigner workflow**, scheduled in parallel by DD (bounded by each model's `max_parallel_requests`). The text-column name and replace strategy are read from the result — you don't pass them again.

Pass either a `PreviewResult` or an `AnonymizerResult`; both carry the trace dataframe, `resolved_text_column`, and `replace_method`.

In [ ]:
evaluated = anonymizer.evaluate(substitute_preview)

In [ ]:
# Same record as before, now annotated with the four LLM-as-a-judge evaluation scores.
evaluated.display_record(12)

## Programmatic access to the verdicts

Each of the four metrics writes two user-facing columns to the trace dataframe:

| Metric | `*_valid` (bool) | `*_invalid_*` (list of flagged items) |
|---|---|---|
| Detection | `detection_valid` | `detection_invalid_entities` |
| Type fidelity | `type_fidelity_valid` | `type_fidelity_invalid_replacements` |
| Relational consistency | `relational_consistency_valid` | `relational_consistency_invalid_relations` |
| Attribute fidelity | `attribute_fidelity_valid` | `attribute_fidelity_invalid_entities` |

A `_valid` cell of `None` means the judge was *Unavailable* for that row (parse failure or the judge skipped it as a trivial passthrough).

In [ ]:
verdict_cols = [
    "detection_valid",
    "type_fidelity_valid",
    "relational_consistency_valid",
    "attribute_fidelity_valid",
]
evaluated.trace_dataframe[verdict_cols].head()

In [ ]:
# Drill into one of the lists — e.g., which entities the detection judge flagged as invalid.
evaluated.trace_dataframe["detection_invalid_entities"].iloc[0]

## Save + reload pattern — iterate on evaluations without re-running detection/replacement

Pickle preserves the full result object (trace dataframe + `resolved_text_column`). Reloading and re-evaluating skips the (expensive) preview step entirely.

In [ ]:
import pickle
from pathlib import Path

cache_path = Path("/tmp/substitute_preview.pkl")

with cache_path.open("wb") as f:
    pickle.dump(substitute_preview, f)

with cache_path.open("rb") as f:
    loaded_preview = pickle.load(f)

# Re-evaluate from the loaded preview — same API, no detection/replacement re-run.
re_evaluated = anonymizer.evaluate(loaded_preview)
re_evaluated.display_record(0)

## Non-Substitute strategies — only Detection Validity is computed

For Annotate / Redact / Hash there is no replacement map, so the type / relational / attribute judges have nothing to evaluate. `evaluate` skips them and runs only the Detection-Validity judge.

In [ ]:
redact_config = AnonymizerConfig(replace=Redact())

redact_preview = anonymizer.preview(
    config=redact_config,
    data=input_data,
    num_records=15,
)

redact_evaluated = anonymizer.evaluate(redact_preview)
redact_evaluated.display_record(0)

In [ ]:
# Only the detection_valid column gets populated. The other three remain absent.
[c for c in redact_evaluated.trace_dataframe.columns if c.endswith("_valid")]